# REPA — Comparison Notebook

Runs all 4 combinations and prints a FID comparison table at the end.

| Encoder | Dataset |
|---|---|
| MedDINOv3 | CIFAR-10 |
| MedDINOv3 | CheXpert |
| DINOv2 | CIFAR-10 |
| DINOv2 | CheXpert |

**Required Kaggle datasets:** `nikiboura/meddinov3-minimal`, `ashery/chexpert`

> Runtime: ~8 hours on T4. Make sure GPU is enabled and internet is on.

## 1. Install dependencies

In [ ]:
%%bash
pip install -q accelerate diffusers timm einops pandas torch-fidelity

## 2. Clone REPA fork

In [ ]:
%%bash
cd /kaggle/working
if [ ! -d "REPA" ]; then
    git clone https://github.com/nikiboura/REPA.git REPA
fi
echo "REPA ready. Commit: $(git -C REPA rev-parse --short HEAD)"

## 3. Set paths

In [ ]:
import os

MEDDINOV3_PATH = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal'
MEDDINOV3_CKPT = MEDDINOV3_PATH + '/checkpoints/model.pth'
CHEXPERT_ROOT  = '/kaggle/input/datasets/ashery/chexpert'

print(f'MedDINOv3 exists  : {os.path.isdir(MEDDINOV3_PATH)}')
print(f'Checkpoint exists : {os.path.isfile(MEDDINOV3_CKPT)}')
print(f'CheXpert exists   : {os.path.isdir(CHEXPERT_ROOT)}')
print(f'train.csv exists  : {os.path.isfile(os.path.join(CHEXPERT_ROOT, "train.csv"))}')

## 4. Prepare CIFAR-10

In [ ]:
import subprocess, sys

result = subprocess.run([
    sys.executable, '/kaggle/working/REPA/prepare_cifar10.py',
    '--out-dir',     '/kaggle/working/data/cifar10_256',
    '--resolution',  '256',
    '--max-samples', '5000',
], text=True)
print('Exit code:', result.returncode)

## 5. Prepare CheXpert

In [ ]:
import subprocess, sys

result = subprocess.run([
    sys.executable, '/kaggle/working/REPA/prepare_chexpert.py',
    '--out-dir',       '/kaggle/working/data/chexpert_256',
    '--chexpert-root', CHEXPERT_ROOT,
    '--resolution',    '256',
    '--max-samples',   '5000',
], text=True)
print('Exit code:', result.returncode)

## 6. Run all combinations
Trains, generates, and computes FID for each combination sequentially.

In [ ]:
import subprocess, os, re, glob
import numpy as np
from PIL import Image
from tqdm import tqdm
from tqdm.notebook import tqdm as tqdm_nb

COMBINATIONS = [
    ('meddinov3', 'cifar10'),
    ('meddinov3', 'chexpert'),
    ('dinov2',    'cifar10'),
    ('dinov2',    'chexpert'),
]

results = {}

for ENCODER, DATASET in COMBINATIONS:
    print(f'\n{"="*50}')
    print(f'  {ENCODER.upper()} x {DATASET.upper()}')
    print(f'{"="*50}')

    EXP_NAME     = f'repa-sits8-{ENCODER}-{DATASET}'
    NUM_CLASSES  = '10' if DATASET == 'cifar10' else '2'
    DATA_DIR     = f'/kaggle/working/data/{DATASET}_256'
    ENC_TYPE     = 'dinov2-vit-b' if ENCODER == 'dinov2' else 'meddinov3-vit-b'
    FID_REAL_DIR = DATA_DIR + ('/images/val' if DATASET == 'cifar10' else '/images/train')

    env = os.environ.copy()
    if ENCODER == 'meddinov3':
        env['MEDDINOV3_PATH'] = MEDDINOV3_PATH
        env['MEDDINOV3_CKPT'] = MEDDINOV3_CKPT

    # ── Train ──────────────────────────────────────────────────
    print('Training...')
    train_cmd = [
        'accelerate', 'launch',
        '--mixed_precision', 'fp16',
        '--num_processes', '1',
        '/kaggle/working/REPA/train.py',
        '--exp-name',            EXP_NAME,
        '--output-dir',          '/kaggle/working/results',
        '--report-to',           'none',
        '--model',               'SiT-S/8',
        '--num-classes',         NUM_CLASSES,
        '--encoder-depth',       '8',
        '--enc-type',            ENC_TYPE,
        '--data-dir',            DATA_DIR,
        '--resolution',          '256',
        '--batch-size',          '32',
        '--num-workers',         '2',
        '--epochs',              '200',
        '--learning-rate',       '1e-4',
        '--mixed-precision',     'fp16',
        '--proj-coeff',          '0.5',
        '--cfg-prob',            '0.1',
        '--path-type',           'linear',
        '--max-train-steps',     '15000',
        '--checkpointing-steps', '15000',
        '--sampling-steps',      '99999999',
    ]
    process = subprocess.Popen(train_cmd, cwd='/kaggle/working/REPA', env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
    pbar = tqdm_nb(total=15000, desc=f'{ENCODER} x {DATASET}')
    last_step = 0
    for line in process.stdout:
        m = re.search(r'Steps:\s*(\d+)/\d+.*?loss=([\d.]+).*?proj_loss=(-?[\d.]+)', line)
        if m:
            step = int(m.group(1))
            if step > last_step:
                pbar.update(step - last_step)
                pbar.set_postfix({'loss': m.group(2), 'proj': m.group(3)})
                last_step = step
    pbar.close()
    process.wait()
    print('Training done.')

    # ── Generate ───────────────────────────────────────────────
    print('Generating 2000 samples...')
    ckpts = sorted(glob.glob(f'/kaggle/working/results/{EXP_NAME}/checkpoints/*.pt'))
    gen_cmd = [
        'torchrun', '--nproc_per_node=1',
        '/kaggle/working/REPA/generate.py',
        '--model',                'SiT-S/8',
        '--ckpt',                 ckpts[-1],
        '--encoder-depth',        '8',
        '--num-classes',          NUM_CLASSES,
        '--projector-embed-dims', '768',
        '--per-proc-batch-size',  '16',
        '--num-fid-samples',      '2000',
        '--path-type',            'linear',
        '--mode',                 'ode',
        '--num-steps',            '50',
        '--cfg-scale',            '1.0',
        '--resolution',           '256',
        '--vae',                  'mse',
    ]
    subprocess.run(gen_cmd, cwd='/kaggle/working/REPA', env=env,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('Generation done.')

    # ── FID ────────────────────────────────────────────────────
    print('Computing FID...')
    gen_npz = sorted(glob.glob('/kaggle/working/REPA/samples/**/*.npz', recursive=True))[-1]
    gen_dir = f'/kaggle/working/gen_{ENCODER}_{DATASET}'
    os.makedirs(gen_dir, exist_ok=True)
    imgs = np.load(gen_npz)[np.load(gen_npz).files[0]]
    for i, img in enumerate(imgs):
        Image.fromarray(img.astype('uint8')).save(f'{gen_dir}/{i:05d}.png')

    fid_out = subprocess.run([
        'fidelity', '--gpu', '0', '--fid',
        '--input1', gen_dir,
        '--input2', FID_REAL_DIR,
    ], capture_output=True, text=True)

    m = re.search(r'frechet_inception_distance:\s*([\d.]+)', fid_out.stdout)
    fid_score = float(m.group(1)) if m else float('nan')
    results[(ENCODER, DATASET)] = fid_score
    print(f'FID: {fid_score}')

print('\nAll combinations done.')

## 7. Results

In [ ]:
print()
print('='*45)
print(f"{'RESULTS':^45}")
print('='*45)
print(f"{'Encoder':<15} {'Dataset':<12} {'FID':>10}")
print('-'*45)
for (enc, ds), fid in results.items():
    print(f'{enc:<15} {ds:<12} {fid:>10.4f}')
print('='*45)